<img src="./images/cads-logo.png" style="height: 100px;" align=left> 
<img src="./images/tf-logo-2.png" style="height: 70px;" align=right>
<img src="./images/keras-logo.png" style="height: 50px;" align=right>

# Convolutional Neural Networks

**Pre-requisites:**

1. Python and Pandas Proficiency 
2. Supervised Machine Learning
3. Basic Matrix Operations
4. Linear Alegbra, Calculus
5. Artificial Neural Networks and Fully Connected/Dense Layers

## Course Outline

1. [Visual Classification with `Dense` Layers](#1.-Visual-Classification-with-Dense-Layers)
1. [Visual Classification with `Conv2D` Layers](#2.-Visual-Classification-with-Conv2D-Layers)
1. [Components of `ConvNet` Architecture](#3.-Components-of-ConvNet-Architecture)
1. [Variation in Visual Classifier Performance](#4.-Variation-in-Visual-Classifier-Performance)


In [ ]:
%load_ext tensorboard

In [ ]:
%matplotlib inline

# 1. Visual Classification with `Dense` Layers

We will build a visual classifier on `fashion_mnist` dataset. <br> 
First , we will make use of purely `Dense` layers and compare its architecture and results to a 2D Convolutional Network  (or `Conv2d`.

<img src="./images/cnn_fc_.png" style="height: 400px;" align=left> 

In [ ]:
import functools

import numpy as np
np.random.seed(42)

import os
import tempfile
import datetime
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (8, 6)
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

from plot_utils import plot_value_array, plotLoss, plotAccuracy

import tensorflow as tf
from tensorflow import keras
tf.keras.backend.clear_session()
from tensorflow.keras import models
from tensorflow.keras import datasets, layers, models, backend
from tensorflow.keras.datasets import cifar10, mnist, fashion_mnist
import tensorflow_docs as tfdocs
import tensorflow_docs.plots
import tensorflow_docs.modeling

In [ ]:
# Question: Which type of tensor data are images?

# SC

Load `fashion_mnist` data. <br> It consists of a training set of 60,000 examples and a test set of 10,000 examples. Each example is a 28x28 grayscale image, associated with a label from 10 classes. `fashion_mnist`, along with `mnist` are widely used datasets for benchmarking maachine learning models. 

Keras API offers an elegant way to split data to `train` and `test` sets. <br>
It will automatically download and cache `fashion_mnist` to disk. <br>

In [ ]:
(train_images, train_labels), (test_images, test_labels) = fashion_mnist.load_data()

In [ ]:
# Define class names
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

In [ ]:
print("Shape: ", train_images[0].shape)
print("Label: ", train_labels[0], "->", class_names[train_labels[0]])

In [ ]:
# Question: What type of classification are we dealing with?

# SC

In [ ]:
train_images.shape, train_labels.shape

In [ ]:
train_labels

In [ ]:
test_images.shape, test_labels.shape

In [ ]:
# Plot sample image data and corresponding label
plt.figure(figsize=(10,10))
for i in range(25):
    plt.subplot(5,5,i+1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(train_images[i], cmap=plt.cm.binary)
    plt.xlabel(class_names[train_labels[i]])
plt.show()

In [ ]:
# Check the pixel value range
plt.figure()
plt.imshow(train_images[0])
plt.colorbar()
plt.grid(False)
plt.show()

Pixel values fall in the range of **`[0-255]`**. <br> 

## Exercise 1:  
1. **Normalize** all the data so that all pixel values fall under **`[0,1]`** before feeding them into the neural network. 

2. State the reason why we need to **normalize the data**.

In [ ]:
# SC

# 1. 

train_images_norm = train_images / ...
test_images_norm = test_images / ...

# 2. 


Extra care should be given in `input_shape`. <br>

In [ ]:
# Keras accecpts the default image data format convention, and it is given by the following
backend.image_data_format()


`channel_first = (depth, height, width)` <br>
`channel_last = (height, width, depth)`

Grayscale images has `depth=1`, while color images has `depth=3`.  <br>
`fashion_mnist` are grayscale images.

In [ ]:
# Convert our train and test sets to `channels_last` formats
train_images_norm = train_images_norm.reshape((train_images_norm.shape[0], 28, 28, 1))
test_images_norm = test_images_norm.reshape((test_images_norm.shape[0], 28, 28, 1))

The **`Flatten`** layer outputs a`1D` by `flattening` the `3D` input. <br>
It simply does the following tensor operation: <br>

 **` (1.1) output = (dim1)(dim2)(dim3)`**
 
 For the case of images: **`dim1 = image_height`**, **`dim2 = image_width`**, and **`dim3 = image_channel`**. <br>
 We can say that `1 pixel` is  `1 feature`.

In [ ]:
# Instantiate modelFC
modelFC = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=(28,28,1)),
    tf.keras.layers.Dense(512, activation='relu'),
    tf.keras.layers.Dense(10, activation='softmax')
])

In [ ]:
# Question : Count the trainable parameters of modelFC. Compare your counted values to modelFC.summar()
# Hint: Connections between layers + bias connections in each layer

# SC

# Flatten Layer: 
# First Dense Layer: 
# Second Dense Layer: 
# Total:


In [ ]:
modelFC.summary()

In [ ]:
# Configure model hyperparameters; Compile
modelFC.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

# Prepare training modelA log storage
logdirFC = os.path.join("logsCNN","modelFC", datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))
tensorboard_callbackFC = tf.keras.callbacks.TensorBoard(logdirFC, histogram_freq=1)

# Train modelFC
historyFC = modelFC.fit(train_images_norm, 
                         train_labels, 
                         epochs=20,
                         batch_size=100,
                         validation_split = 0.2,  
                         callbacks=[tensorboard_callbackFC])

# Evaluate modelFC in test set
test_loss, test_acc = modelFC.evaluate(test_images_norm,  test_labels, verbose=2)
print('\nTest accuracy:', test_acc)

In [ ]:
modelFC.save('models/modelFC', save_format='tf' )

In [ ]:
plotLoss(historyFC, 'modelFC', 'modelFC Loss', 'modelFC Val Loss')

In [ ]:
plotAccuracy(historyFC, 'modelFC', 'modelFC Accuracy', 'modelFC Val Accuracy')

With the model trained, you can use it to make predictions on the test images. <br>

In [ ]:
predictions = modelFC.predict(test_images_norm)

In [ ]:
# Check position (label) of highest probability
np.argmax(predictions[0])

A prediction is an array of 10 numbers. They represent the model's "confidence" that the image corresponds to each of the 10 different articles of clothing. You can see which label has the highest confidence value:

In [ ]:
def plot_image(i, predictions_array, true_label, img):
    predictions_array, true_label, img = predictions_array, true_label[i], img[i]
    plt.grid(False)
    plt.xticks([])
    plt.yticks([])

    plt.imshow(img)

    predicted_label = np.argmax(predictions_array)
    if predicted_label == true_label:
        color = 'blue'
    else:
        color = 'red'

    plt.xlabel("{} {:2.0f}% ({})".format(class_names[predicted_label],
                                100*np.max(predictions_array),
                                class_names[true_label]),
                                color=color)

In [ ]:
# Plot the first X test images, their predicted labels, and the true labels.
# Color correct predictions in blue and incorrect predictions in red.
num_rows = 5
num_cols = 3
num_images = num_rows*num_cols
plt.figure(figsize=(2*2*num_cols, 2*num_rows))
for i in range(num_images):
    plt.subplot(num_rows, 2*num_cols, 2*i+1)
    plot_image(i, predictions[i], test_labels, test_images_norm.squeeze())
    plt.subplot(num_rows, 2*num_cols, 2*i+2)
    plot_value_array(i, predictions[i], test_labels)
plt.tight_layout()
plt.show()

# 2. Visual Classification with `Conv2D` Layers

We can also solve `fashion_mnist` classification problem using **Convolutional Neural Networks `(ConvNet)`**.<br>Observe first how these layers are added and stacked on top of the other;  also how  **`Conv2D`** and **`MaxPooling2D`** layers are  used within the network. <br> 

In [ ]:
# Instantiate modelFC
modelCNN = tf.keras.Sequential([
    
                    tf.keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1), padding='SAME', strides=1),
                    tf.keras.layers.MaxPooling2D((2, 2)),
    
                    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
                    tf.keras.layers.MaxPooling2D((2, 2)),
    
                    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    
                    tf.keras.layers.Flatten(),
    
                    tf.keras.layers.Dense(64, activation='relu'),
                    tf.keras.layers.Dense(10, activation='softmax')
])


In [ ]:
modelCNN.summary()


In [ ]:
modelCNN.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

In [ ]:
# Prepare training modelCNN log storage
logdirCNN = os.path.join("logsCNN","modelCNN", datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))
tensorboard_callbackCNN = tf.keras.callbacks.TensorBoard(logdirCNN, histogram_freq=1)

# Train modelFC
historyCNN = modelCNN.fit(train_images_norm, 
                          train_labels, 
                          epochs=20, 
                          batch_size=100,
                          validation_split = 0.2,  
                          callbacks=[tensorboard_callbackCNN])


# Evaluate modelCNN in test set
test_loss, test_acc = modelCNN.evaluate(test_images_norm,  test_labels, verbose=2)
print('\nTest accuracy:', test_acc)

In [ ]:
modelCNN.save('models/modelCNN' , save_format='tf')

In [ ]:
plotLoss(historyCNN, 'modelCNN', 'modelCNN Loss', 'modelCNN Val Loss')
plotAccuracy(historyCNN, 'modelCNN', 'modelCNN Loss', 'modelCNN Val Loss')

In [ ]:
# Evaluate modelCNN in test set
test_loss, test_acc = modelCNN.evaluate(test_images_norm,  test_labels, verbose=2)
print('\nTest accuracy:', test_acc)

In [ ]:
predictionsCNN = modelCNN.predict(test_images_norm)

In [ ]:
# Plot the first X test images, their predicted labels, and the true labels.
# Color correct predictions in blue and incorrect predictions in red.
num_rows = 5
num_cols = 3
num_images = num_rows*num_cols
plt.figure(figsize=(2*2*num_cols, 2*num_rows))
for i in range(num_images):
    plt.subplot(num_rows, 2*num_cols, 2*i+1)
    plot_image(i, predictionsCNN[i], test_labels, test_images_norm.squeeze())
    plt.subplot(num_rows, 2*num_cols, 2*i+2)
    plot_value_array(i, predictionsCNN[i], test_labels)
plt.tight_layout()
plt.show()

# 3. Components of `ConvNet` Architecture

## 3.1. `Conv2D` Layer

The convolution layer **`Conv2D`** uses filters that perform convolution operations as it is scanning the input 
**`I`** with respect to its dimensions. Its hyperparameters include the` filter` (or `kernel`) size **`F`** and stride **`S`**. The resulting output , **`O`** is called feature map or activation map. In this case, we  configured the `Conv2D`  to process inputs of size `(28, 28, 1)`.
```python 
tf.keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1), strides=1)`**
```
The parameter **`32`** indicates the **number of `filters` or `kernel`** <br>
The parameter **`(3,3)`** indicates the **size of each `filter`** <br>


Displayed below is the architecture of our `modelCNN`.

In [ ]:
modelCNN.summary()

The output of every `Conv2D` and `MaxPooling2D` layers are 3D tensors having dimensions of **`(height, width, channels)`** that tend to **_shrink_** as we go deeper into the model. The number of `channels` is set by the **number of `filters`**. 

### `Dense` Layer Learns _Global_ Patterns while `Conv2D` Layer Learns _Local_ Patterns

**`Dense`** layers:
- learn **global patterns** in their input feature space. In the case of `fashion_mnist`, patterns involve all pixels. 



**`Conv2D`** layers:

- learn **local patterns**. These patterns are **translation invariant**. Aftet learning a localized pattern, the `Conv2D` can recognize it anywhere in the image. A densely connected network would have to learn the pattern anew if it appeared at a new location. This makes `Conv2D` data efficient when processing images as it require fewer training samples to generalized representations. 

- learn spatial hierarchies of patterns from simple to more complex: e.g. first Conv2D layer learns edges; second Conv2D layer learns contours, etc.

- The feature map **`O`** is still a `3D tensor` having `width`, `height`, and `depth`. 

<img src="./images/cnn_textures.png" style="height: 300px;" align=left> 

Consider a `Conv2D` layer specified below. For simplicity, we drop the activation function.  We will show that it's output has size **`(28,28,3)`**. 

```python 
tf.keras.layers.Conv2D(3, (3, 3), input_shape=(28, 28, 1))`**
```
Here we have `filters=3` with size `(3,3,1)`. Filters always extend the full volume of the input.

--- 

 The first **`filter w1`**  having the size of **`(3, 3, 1)`** takes input **feature map X** of size **`(28, 28, 1)`**. The response of the filter over this input is equal to 

  **` w1_output = dot(w1, X) + b`**
  

**`filter w1`** is slid (**convolved**) over all possible spatial locations. The result of this convolution is an output **feature map** of size **`(26, 26, 1)`**.

<img src="./images/cnn_f1.png" style="height: 400px;" align=left> 

If we have a total of **`3 filters: w1, w2, w3`**, each of the filters has an output *feature map* of size (26, 26, 1), representing its  response at different locations of the input. 

<img src="./images/cnn_f1.png" style="height: 200px;" align=left> 
<img src="./images/cnn_f2.png" style="height: 200px;" align=left> 
<img src="./images/cnn_f3.png" style="height: 200px;" align=left> 

 We stack the `3` output feature maps to get a **'new'** image of size **`(26,26,3)`**. 

<img src="./images/cnn_channels.png" style="height: 450px;" align=center> 

Other paramaters that can be specified in a `Conv2D` layer are:

- **`strides`** – an integer or tuple of 2 integers, specifying how the filter should move along the height and width. Set a single integer to use the same stride value for both dimensions.

- **`padding`**: 
    - **`"VALID"`**,  image is padded with a border of one pixel, or 
    - **`"SAME"`**, filter moves within the image with **no padding**, generating a smaller output.

#### 3.a.1. `SAME` : No Padding

**`O = ((I - F)/S) + 1`**

`I: input length; F: filter length; S: stride length`


**` I = (6 , 6, 1)`** <br>
**` F =  (3 , 3, 1)`**


> `(6 - 3 )/1 + 1 = 4`

**` O = (4 , 4, 1)`**

<img src="./images/conv2d_no_padding.png" style="height: 300px;" align=left> 

Image Credit ([Source](https://stanford.edu/~shervine/teaching/cs-230/cheatsheet-convolutional-neural-networks))

In `modelCNN` first `Conv2D` layer, has the following output feature map size:

> **`(28 - 3)/1 + 1 = 26`**


#### 3.a.2. `VALID`: With Padding


It is common to `zero`-pad input feature maps.  


**` I = (7, 7, 1)`** <br>
**` F =  (3 , 3, 1)`**


> `(7 - 3 )/1 + 1 = 4`

**` O = (5 , 5, 1)`**

<img src="./images/conv2d_padding.gif" style="height: 300px;" align=left> 

This animation was accessed from [StackOverflow](https://stackoverflow.com/questions/52067833/how-to-plot-an-animated-matrix-in-matplotlib).

## Exercise 2:

1. Construct  `modelConv_1` with a single `Conv2D` layer

`input_shape=(32,32,1)`<br>
`filter_size=(5,5)` <br>
`filters=10`, <br>
`padding='SAME'`, <br>
`strides=1'`

2. What is the output feature map shape? <br>
    Recall: ` O = ((I - F)/S ) + 1` 
    
    
3. Count the trainable parameters for this particular model. Compare with `modelConv_1.summary()`results.

In [ ]:
# SC 

# 1. 

modelConv_1 = tf.keras.Sequential([ 
    tf.keras.layers.Conv2D(..., (...), input_shape=(...), strides=..., padding=...)])


# 2. 


# 3. 

# Filter volume = (5, 5, 1), since filter extends the volume of input

modelConv_1.summary()

## Exercise 3:

1. Construct  `modelConv_2` with a single `Conv2D` layer

`input_shape=(32,32,3)`<br>
`filter_size=(5,5)` <br>
`filters=10`, <br>
`padding='SAME'`, <br>
`strides=1'`

2. What is the output feature map shape? Hint: this time the input feature map has `volume`. <br>
    Recall: ` O = ((I - F)/S ) + 1` and the filters always extend the full volume of the input.
    
    
3. Count the trainable parameters for this particular model. Compare with `modelCOnv_1.summary()`results.

In [ ]:
# SC 

# 1. 

model_2 = tf.keras.Sequential([ 
    tf.keras.layers.Conv2D(..., (...), input_shape=(...), strides=..., padding=...)])


# 2. 


# 3. 
# Filter volume = (5, 5, 3), since filter extends the volume of input


model_2.summary()

## 3.1.1. Visualizing `Conv2D` outputs

### 3.1.1.1.  Visualizing Intermediate Activations

Let's use our previously trained and saved **`modelCNN4`**. 

In [ ]:
modelCNN_new = keras.models.load_model('models/modelCNN4')
# Select sample image from test set
img = test_images[200]
type(img), img.shape
img.shape

# Check the pixel value range
plt.figure()
plt.imshow(img)
plt.colorbar()
plt.grid(False)
plt.show()

In [ ]:
layer_outputs = [layer.output for layer in modelCNN_new.layers[:0]] 
activation_model = models.Model(inputs=modelCNN_new.input, outputs=layer_outputs)
activations = activation_model.predict(img)
first_layer_activation = activations[0]
# 4th filter activation/feture map of the first layer
plt.matshow(first_layer_activation[0, :, :, 4], cmap='viridis')

In [ ]:
layer_names = []
for layer in modelCNN_new.layers[:1]:
    layer_names.append(layer.name)
images_per_row = 16

for layer_name, layer_activation in zip(layer_names, activations):
    n_features = layer_activation.shape[-1]
    size = layer_activation.shape[1]
    n_cols = n_features // images_per_row
    display_grid = np.zeros((size * n_cols, images_per_row * size))
    for col in range(n_cols):
        for row in range(images_per_row):
            channel_image = layer_activation[0,:, :,col * images_per_row + row]
            channel_image -= channel_image.mean()
            channel_image /= channel_image.std()
            channel_image *= 64
            channel_image += 128
            channel_image = np.clip(channel_image, 0, 255).astype('uint8')
            display_grid[col * size : (col + 1) * size,
             row * size : (row + 1) * size] = channel_image
scale = 1. / size
plt.figure(figsize=(scale * display_grid.shape[1],
                    scale * display_grid.shape[0]))
plt.title(layer_name)
plt.grid(False)
plt.imshow(display_grid, aspect='auto', cmap='viridis')

### 3.1.1.2. Visualizing Filters

Each `filter/kernel` in a `Conv2D` layer *responds* to a particular pattern. It is possible to see what each layer does by visualizing these patterns. **`Gradient Ascent`** is a method of finding the `input_image` that the `filter` will `maximally` respond to.  

The process starts  from a blank `input_image`. In contrast to `Gradient Descent`, the goal is to build a loss function that maximizes the value of the filter in `Conv2D` layer. `SGD` will  help in adjusting the values of the `input_image` that maximizes the activation value. 

We will use the pre-trained weights of `VGG19`.

In [ ]:
# Layer name to inspect
layer_name = 'block3_conv1'
step_size = 1.
filter_index = 0

def generate_pattern(layer_name, filter_index, epochs, size):
    # Create a connection between the input and the target layer
    model = tf.keras.applications.vgg19.VGG19(weights='imagenet', include_top=False)
    submodel = tf.keras.models.Model([model.inputs[0]], [model.get_layer(layer_name).output])

    # Initiate random noise
    input_img_data = np.random.random((1, size, size, 3))
    input_img_data = (input_img_data - 0.5) * 20 + 128.

    # Cast random noise from np.float64 to tf.float32 Variable
    input_img_data = tf.Variable(tf.cast(input_img_data, tf.float32))

    # Gradient ascents loop

    for _ in range(epochs):
        with tf.GradientTape() as tape:
            outputs = submodel(input_img_data)
             # Compute the value of the loss tensor
            loss_value = tf.reduce_mean(outputs[:, :, :, filter_index])
        # Compute the gradient tensor, given an input image
        grads = tape.gradient(loss_value, input_img_data)
        normalized_grads = grads / (tf.sqrt(tf.reduce_mean(tf.square(grads))) + 1e-5)
        input_img_data.assign_add(normalized_grads * step_size)
    img = input_img_data[0]
    return img

In [ ]:
# Preprocess the resulting image tensor to allow visualization
img = generate_pattern(layer_name, filter_index,  epochs= 50,  size=224)
img = np.clip(img, 0, 255).astype('uint8')

In [ ]:
plt.imshow(img)

`filter 0` in layer `block3_conv1` seems responsive to a polka-dot pattern. 

Let us visualize every filter in every layer.

In [ ]:
layer_name = 'block1_conv1'
# Look only at the first 64 filters
size = 64
margin = 5
filter_index = 0

# Empty (black) image to store results
results = np.zeros((8 * size + 7 * margin, 8 * size + 7 * margin, 3))
for i in range(8): #Iterates over the rows of the results grid
    for j in range(8): #Iterates over the columns of the results grid
        filter_img = generate_pattern(layer_name, filter_index,  epochs= 50,  size=64)
        # Preprocess the resulting image tensor to allow visualization
        filter_img = np.clip(filter_img, 0, 255).astype('uint8')
        #Generate the pattern for filter i + (j * 8) in layer_name
        horizontal_start = i * size + i * margin
        horizontal_end = horizontal_start + size
        vertical_start = j * size + j * margin
        vertical_end = vertical_start + size
        results[horizontal_start: horizontal_end,
        vertical_start: vertical_end, :] = filter_img

In [ ]:
plt.figure(figsize=(40, 40))
plt.imshow(results)

## 3.2. `Pooling` Layer

It is common to periodically insert a `MaxPooling2d` layer  in-between successive `Conv2D` layers in a `ConvNet` architecture. Its function is to progressively reduce the spatial size of the representation (`downsample`)  to reduce the amount of parameters and computation in the network, and hence to **control overfitting**. 

### 3.2.1. `MaxPooling2D` Layer

```python 
tf.keras.layers.MaxPooling2D((2, 2))

```

The `MaxPooling2D` layer operates independently on every depth slice of the input and resizes it spatially, using the `max` operation. The most common form is a pooling layer with filters of size `(2,2)` applied with a stride of `2`. It downsamples every depth slice in the input by `2` along both `width` and `height`, discarding `75%` of the activations. Every `max` operation would in this case be taking a `max` over `4` numbers (little `2x2` region in some depth slice) as shown below. The depth dimension remains unchanged.

<img src="./images/maxpool_b.png" style="height: 300px;" align=left> 

Image Credit ([Source](https://stanford.edu/~shervine/teaching/cs-230/cheatsheet-convolutional-neural-networks))

In [ ]:
#We can write the `MaxPooling2D` layer above as:

modelMaxPool_1 = tf.keras.Sequential([tf.keras.layers.MaxPooling2D((2, 2), input_shape=(6, 6, 1 ))])
modelMaxPool_1.summary()

### 3.2.2. `AveragePooling2D` Layer


````python 
tf.keras.layers.AveragePooling2D((2, 2))
````

Instead of using the `max` function, the `AveragePooling2D` gets the average of the `(2,2)` region above. 

In [ ]:
#We can write the `AveragePooling2D` layer as:

modelAvePool_1 = tf.keras.Sequential([tf.keras.layers.AveragePooling2D((2, 2), input_shape=(6, 6, 1 ))])
modelAvePool_1.summary()

# Exercise 4:

1. Go back to `modelCNN`. Instead of MaxPooling2D, use `AveragePooling2D`. Name your new model **`modelCNN_Ave`**. <br>
 
2. Optional: Count the number of trainable parameters for each layer. Check it with `modelCNN_Ave.summary()`.
3. Train the model and assign results to `historyCNN_Ave`.
4. Plot both `loss` and `accuracy` and compare it with `modelCNN`.

In [ ]:
# MC

# 1. 
modelCNN_... = tf.keras.Sequential([
    
                    tf.keras.layers.Conv2D(32, (5, 5), activation='relu', input_shape=(28, 28, 1), padding='SAME', strides=1),
                    ...,
    
                    tf.keras.layers.Conv2D(64, (5, 5), activation='relu'),
                    ...,
    
                    tf.keras.layers.Conv2D(64, (5, 5), activation='relu'),
    
                    tf.keras.layers.Flatten(),
    
                    tf.keras.layers.Dense(64, activation='relu'),
                    tf.keras.layers.Dense(10, activation='softmax')
])


In [ ]:
# MC

# 2. 
modelCNN_....summary()

In [ ]:
# MC

# 3. 
modelCNN_....compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

# Train modelFC
historyCNN_... = modelCNN_....fit(...)

# Evaluate modelFC in test set
test_loss, test_acc = ...
print('\nTest accuracy:', test_acc)

In [ ]:
plotLoss(historyCNN_Ave, 'modelCNN_Ave', 'modelCNN_Ave Loss', 'modelCNN_Ave Val Loss')
plotAccuracy(historyCNN_Ave, 'modelCNN_Ave', 'modelCNN_Ave Acc', 'modelCNN_Ave Val Acc')

In [ ]:
predictionsCNN_... = modelCNN_....predict(test_images_norm)

In [ ]:
# Plot the first X test images, their predicted labels, and the true labels.
# Color correct predictions in blue and incorrect predictions in red.
num_rows = 5
num_cols = 3
num_images = num_rows*num_cols
plt.figure(figsize=(2*2*num_cols, 2*num_rows))
for i in range(num_images):
    plt.subplot(num_rows, 2*num_cols, 2*i+1)
    plot_image(i, predictionsCNN_...[i], test_labels, test_images_norm.squeeze())
    plt.subplot(num_rows, 2*num_cols, 2*i+2)
    plot_value_array(i, predictionsCNN_...[i], test_labels)
plt.tight_layout()
plt.show()

## 3.3. `Flatten` Layer and `Dense` Layer

The **`Flatten`** layer outputs a `vector (1D tensor)` by `flattening` the `3D` input. <br>
It simply does the following tensor operation: <br>

 **` (3.3) output = dim1*dim2*dim3`**
 
 For the case of images: **`dim1 = height`**, **`dim2 = width`**, and **`dim3 = channel`**. <br>
 We can say that `1 pixel` is  `1 feature`.

```python
tf.keras.layers.Flatten()
tf.keras.layers.Dense(64, activation='relu'),
tf.keras.layers.Dense(10, activation='softmax')
        ```

Recall that the `Dense` layer processes `1D` tensors. In this CNN the `Dense` layer operates on a `Flatten` input where each input is connected to all neurons. If present, `Dense` layers are usually found towards the end of CNN architectures and are used to optimize objectives such as class scores.

<img src="./images/fc_after_conv.png" style="height: 200px;" align=left> 

Image Credit ([Source](https://stanford.edu/~shervine/teaching/cs-230/cheatsheet-convolutional-neural-networks))

# Exercise 5:

1. Go back to `modelCNN`. Name your new model **`modelCNN2`**. <br>
   Instead of a `(3,3)` filter, use **`(5,5)`** filter in all `Conv2D` layers. 
 
2. Optional: Count the number of trainable parameters for each layer. Check it with `modelCNN2.summary()`.
3. Train the model and assign results to `historyCNN2`.
4. Plot both `loss` and `accuracy` and compare it with `modelCNN`.

In [ ]:
# MC

# 1. 
... = tf.keras.Sequential([
    
                    tf.keras.layers.Conv2D(32, (...), activation='relu', input_shape=(28, 28, 1), padding='SAME', strides=1),
                    tf.keras.layers.MaxPooling2D((2, 2)),
    
                    tf.keras.layers.Conv2D(64, (...), activation='relu'),
                    tf.keras.layers.MaxPooling2D((2, 2)),
    
                    tf.keras.layers.Conv2D(64, (...), activation='relu'),
    
                    tf.keras.layers.Flatten(),
    
                    tf.keras.layers.Dense(64, activation='relu'),
                    tf.keras.layers.Dense(10, activation='softmax')
])


In [ ]:
# MC


# 2. 




modelCNN2.summary()

In [ ]:
# MC

# 3. 
....compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

# Prepare training modelCNN log storage
... = os.path.join("logsCNN","modelCNN2", datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))
.. = tf.keras.callbacks.TensorBoard(..., histogram_freq=1)

# Train modelFC
... = modelCNN2.fit(train_images_norm, 
                          train_labels, 
                          epochs=20, 
                          batch_size=100,
                          validation_split = 0.2,  
                          callbacks=[...])

# Evaluate modelFC in test set
test_loss, test_acc = ....evaluate(test_images_norm,  test_labels, verbose=2)
print('\nTest accuracy:', test_acc)

In [ ]:
....save('models/...', save_format='tf')

In [ ]:
plotLoss(historyCNN2, 'modelCNN2', 'modelCNN2 Loss', 'modelCNN2 Val Loss')
plotAccuracy(historyCNN2, 'modelCNN2', 'modelCNN2 Acc', 'modelCNN2 Val Acc')

In [ ]:
predictionsCNN2 = ....predict(test_images_norm)

In [ ]:
# Plot the first X test images, their predicted labels, and the true labels.
# Color correct predictions in blue and incorrect predictions in red.
num_rows = 5
num_cols = 3
num_images = num_rows*num_cols
plt.figure(figsize=(2*2*num_cols, 2*num_rows))
for i in range(num_images):
    plt.subplot(num_rows, 2*num_cols, 2*i+1)
    plot_image(i, predictionsCNN2[i], test_labels, test_images_norm.squeeze())
    plt.subplot(num_rows, 2*num_cols, 2*i+2)
    plot_value_array(i, predictionsCNN2[i], test_labels)
plt.tight_layout()
plt.show()

# 4. Variation in Visual Classifier Performance

## 4.1. Adding Drop-out Layers

# Exercise 6:

1. Create a new **`modelCNN3`**  based from `modelCNN2`. Add `Dropout` layer every after `MaxPooling2D`.
        
    `Dropout(0.2)`


2. Find out the number of trainable parameters for each layer and compare to `modelCNN3.summary()`.
3. Train the model and assign results to `historyCNN3`.
4. Plot both `loss` and `accuracy` and compare it with `modelCNN2`.


In [ ]:
# MC


# 1. 
.. = tf.keras.Sequential([
    
                    ...
                    ...,
                    ...,
                    ...,
                    ...,
                    ...,
    
                    tf.keras.layers.Flatten(),
                    tf.keras.layers.Dense(64, activation='relu'),
                    tf.keras.layers.Dense(10, activation='softmax')
])

In [ ]:
....summary()

In [ ]:
....compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

In [ ]:
# Prepare training modelCNN log storage
... = os.path.join("logsCNN","modelCNN3", datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))
... = tf.keras.callbacks.TensorBoard(..., histogram_freq=1)

# Train modelFC
... = ....fit(train_images_norm, 
                          train_labels, 
                          epochs=20, 
                          batch_size=100,
                          validation_split = 0.2,  
                          callbacks=[...])

# Evaluate modelFC in test set
test_loss, test_acc = ...evaluate(test_images_norm,  test_labels, verbose=2)
print('\nTest accuracy:', test_acc)

In [ ]:
....save('models/modelCNN3', save_format='tf')

In [ ]:
plotLoss(historyCNN3, 'modelCNN3', 'modelCNN3 Loss', 'modelCNN3 Val Loss')
plotAccuracy(historyCNN3, 'modelCNN3', 'modelCNN3 Acc', 'modelCNN3 Val Acc')

In [ ]:
predictionsCNN3 = ....predict(test_images_norm)

In [ ]:
# Plot the first X test images, their predicted labels, and the true labels.
# Color correct predictions in blue and incorrect predictions in red.
num_rows = 5
num_cols = 3
num_images = num_rows*num_cols
plt.figure(figsize=(2*2*num_cols, 2*num_rows))
for i in range(num_images):
    plt.subplot(num_rows, 2*num_cols, 2*i+1)
    plot_image(i, predictionsCNN3[i], test_labels, test_images_norm.squeeze())
    plt.subplot(num_rows, 2*num_cols, 2*i+2)
    plot_value_array(i, predictionsCNN3[i], test_labels)
plt.tight_layout()
plt.show()

## 4.2. Reduce `model` complexity

# Exercise 7:

1. Go back to `modelCNN3`. Remove the last set of `Conv2D` + `MaxPooling2D` + `Dropout` layers. Call this **`modelCNN4`**. 

2. Find out the number of trainable parameters for each layer and compare to `modelCNN4.summary()`.
3. Train the model and assign results to `historyCNN4`.
4. Plot both `loss` and `accuracy` and compare it with `modelCNN3.`.


In [ ]:
# MC


# 1. 
... = tf.keras.Sequential([
    
                    ...,
                    ...,
                    ...,

                    ...,
    
                    ...,
                    ...
])

In [ ]:
....summary()

In [ ]:
....compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

In [ ]:
# Prepare training modelCNN log storage
... = os.path.join("logsCNN","modelCNN3", datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))
... = tf.keras.callbacks.TensorBoard(..., histogram_freq=1)

# Train modelFC
... = ....fit(train_images_norm, 
                          train_labels, 
                          epochs=20, 
                          batch_size=100,
                          validation_split = 0.2,  
                          callbacks=[...])

# Evaluate modelFC in test set
test_loss, test_acc = ....evaluate(test_images_norm,  test_labels, verbose=2)
print('\nTest accuracy:', test_acc)

In [ ]:
....save('models/modelCNN4',  save_format='tf')

In [ ]:
plotLoss(historyCNN4, 'modelCNN4', 'modelCNN4 Loss', 'modelCNN4 Val Loss')
plotAccuracy(historyCNN4, 'modelCNN4', 'modelCNN4 Acc', 'modelCNN4 Val Acc')

In [ ]:
predictionsCNN4 = modelCNN4.predict(test_images_norm)

In [ ]:
# Plot the first X test images, their predicted labels, and the true labels.
# Color correct predictions in blue and incorrect predictions in red.
num_rows = 5
num_cols = 3
num_images = num_rows*num_cols
plt.figure(figsize=(2*2*num_cols, 2*num_rows))
for i in range(num_images):
    plt.subplot(num_rows, 2*num_cols, 2*i+1)
    plot_image(i, predictionsCNN4[i], test_labels, test_images_norm.squeeze())
    plt.subplot(num_rows, 2*num_cols, 2*i+2)
    plot_value_array(i, predictionsCNN4[i], test_labels)
plt.tight_layout()
plt.show()

## 4.3. Increase training loops

# Exercise 8:

1. Use **`modelCNN3`**. 
2. Train the model with `EPOCHS=50` and assign to `historyCNN3_ep50`.
3. Plot both `loss` and `accuracy` and compare it with `modelCNN3` ran in `20` epochs. 


In [ ]:
# Prepare training modelCNN log storage

# Train modelFC
...= modelCNN3.fit(train_images_norm, 
                          train_labels, 
                          epochs=..., 
                          batch_size=100,
                          validation_split = 0.2,  
                          callbacks=[tensorboard_callbackCNN3])

# Evaluate modelFC in test set
test_loss, test_acc = modelCNN3.evaluate(test_images_norm,  test_labels, verbose=2)
print('\nTest accuracy:', test_acc)

In [ ]:
plotLoss(historyCNN3_ep50, 'modelCNN3 + 50 epochs', 'modelCNN3 Loss', 'modelCNN3 Val Loss')
plotAccuracy(historyCNN3_ep50, 'modelCNN3 + 50 epochs', 'modelCNN3 Acc', 'modelCNN3 Val Acc')

In [ ]:
predictionsCNN3_ep50 = modelCNN3.predict(test_images_norm)

In [ ]:
# Plot the first X test images, their predicted labels, and the true labels.
# Color correct predictions in blue and incorrect predictions in red.
num_rows = 5
num_cols = 3
num_images = num_rows*num_cols
plt.figure(figsize=(2*2*num_cols, 2*num_rows))
for i in range(num_images):
    plt.subplot(num_rows, 2*num_cols, 2*i+1)
    plot_image(i, predictionsCNN3_ep50[i], test_labels, test_images_norm.squeeze())
    plt.subplot(num_rows, 2*num_cols, 2*i+2)
    plot_value_array(i, predictionsCNN3_ep50[i], test_labels)
plt.tight_layout()
plt.show()

In [ ]:
%tensorboard --logdir=logsCNN

# References

Majority of discussions are based from "Deep Learning with Python" by François  Chollet. <br>
However, contents are modified to accommodate Keras in Tensorflow 2.0 Backend. <br>
Images ours, unless otherwise specified. 

1. "Classification on imbalanced data." Tensorflow. Apr 04, 2020,  https://www.tensorflow.org/tutorials/structured_data/imbalanced_data. Accessed 12 April 2020.

1. Karpathy, Andrej. "A Recipe for Training Neural Networks." Andrej Karpathy blog, Apr 25, 2019, http://karpathy.github.io/2019/04/25/recipe/#2-set-up-the-end-to-end-trainingevaluation-skeleton--get-dumb-baselines. Accessed 12 April 2020.

1. Chollet, François. Deep Learning with Python. Manning, 2018.

1. Chollet, François. "Transfer learning with a pretrained ConvNet." Tensorflow. Apr 04, 2020,  https://www.tensorflow.org/tutorials/images/transfer_learning. Accessed 15 April 2020.

1. Karen Simonyan and Andrew Zisserman, “Very Deep Convolutional Networks for Large-Scale Image Recognition,” arXiv (2014), https://arxiv.org/abs/1409.1556.

1. "Convolutional Neural Networks Cheatsheet."  CS 230 - Deep Learning. https://stanford.edu/~shervine/teaching/cs-230/cheatsheet-convolutional-neural-networks. Accessed 19 October 2019. 
